# 📗 모델 서빙 API: FastAPI로 vLLM 서빙 API 만들기

<div class="day77-callout" style="margin:10px 0;padding:8px 14px;font-size:17px;line-height:1.75;color:#202124;background:#ffffff;border-left:4px solid #54769b;border-radius:4px;"><strong style="display:block;font-size:1.13em;color:#1f3a5f;margin-bottom:2px;">실습 환경</strong><strong style="font-size:1em;color:#1f3a5f;">실행:</strong> RunPod의 Linux + NVIDIA GPU에서 vLLM 0.27.0 서버, FastAPI 서버, 노트북을 모두 실행합니다. <strong style="font-size:1em;color:#1f3a5f;">내 PC에서 실행하는 코드는 없습니다.</strong> VS Code 창은 내 PC에 열리지만, 그 안의 노트북과 터미널은 RunPod에서 실행됩니다.<br><strong style="font-size:1em;color:#1f3a5f;">접속:</strong> VS Code Remote SSH로 파드에 접속해 원격 노트북과 터미널을 사용합니다. 노트북은 같은 파드의 FastAPI 서버(127.0.0.1:8080)에 요청을 보냅니다.<br><strong style="font-size:1em;color:#1f3a5f;">준비:</strong> 딥러닝 학습 원리 단원 <a style="color:#245fa0;text-decoration:underline;" href="../../day65_딥러닝_학습원리/부록_RunPod_GPU에서_실습하기.ipynb">RunPod 부록 5~7절</a>에 따라 원격 저장소를 열고, 저장소의 .venv 커널을 선택합니다.<br><strong style="font-size:1em;color:#1f3a5f;">최소 권장:</strong> GPU VRAM 24GB 1장, RAM 16GB, 디스크 80GB. vLLM 서버는 하나만 실행합니다.</div>

#### **모델 서빙 API**

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">모델 서버 앞에 두어, 여러 앱이 정해진 주소와 요청 양식으로 모델을 쓰게 하는 웹 API</strong>입니다.</p>
<p style="margin:6px 0;">vLLM 서버는 답 토큰을 만드는 계산을 맡고, 누가 어떤 양식으로 얼마나 쓰는지는 서빙 API가 정합니다. vLLM 서버에도 API가 있지만, 그 주소를 앱에 바로 열지 않고 FastAPI를 앞에 두면 다음 다섯 가지가 달라집니다.</p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">항목</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">vLLM 주소를 바로 열면</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">FastAPI를 앞에 두면</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">키</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">서버를 키 하나로 띄우므로 모든 사용자가 같은 키를 씁니다</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">사용자마다 키를 발급하고, 한 사용자만 끊을 수 있습니다(교안 02, 03)</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">요청 내용</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">사용자가 시스템 메시지와 생성 상한(답으로 만들 최대 토큰 수)까지 정해 보냅니다</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">서비스가 정한 시스템 메시지와 생성 상한을 서버에서 붙이고, 사용자는 질문만 보냅니다</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">사용량</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">한 사용자가 요청을 몰아 보내도 막을 수 없습니다</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">사용자별 요청 한도를 넘으면 429로 돌려보냅니다(교안 02)</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">오류</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">vLLM이 꺼지면 연결 오류만 남습니다</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">정해진 오류 양식(502, 504)으로 돌려줍니다(교안 02)</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">기록</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">누가 얼마나 썼는지 알 수 없습니다</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">응답마다 토큰 사용량을 돌려주고, 키별 사용량을 DB에 기록합니다(교안 03)</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;">상담 봇이나 사내 검색 도우미처럼 여러 화면과 서비스가 모델 하나를 함께 쓸 때 이 구조를 씁니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="../images/00_serving_api_concept.png" width="1050" alt="여러 앱의 요청을 FastAPI 서빙 API가 받아 vLLM 서버로 보내고, vLLM이 만든 답을 다시 앱으로 돌려주는 구조입니다.">

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">이 교안에서는 교안 00에서 익힌 FastAPI 기초(엔드포인트, 요청 본문 스키마, async def, 스트리밍 응답)를 이어, vLLM 추론 엔진 단원에서 띄운 vLLM 서버 앞에 <strong style="font-size:1.13em;color:#1f3a5f;">FastAPI로 노무 상담 서빙 API를 세웁니다.</strong> 같은 RunPod 파드에서 FastAPI 서버가 vLLM에 요청하고, 원격 노트북은 FastAPI 서버에만 요청합니다. 교안 02는 이 API에 인증, 요청 한도, 오류 처리를 붙입니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="../images/10_lesson01_goals.png" width="1050" alt="이번 교안의 학습 목표 네 가지를 번호 순서로 놓은 판 네 개입니다. 서빙 API를 패키지와 주소 묶음으로 나누기, vLLM 백엔드에 연결하기, 여러 요청을 함께 보내기, 답 조각을 중계하기를 보여 줍니다.">

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">

<span class="day77-goal" style="font-size:1.13em;color:#1f3a5f;">**학습 목표**</span> (그림의 번호와 같습니다)

1. 서빙 API를 계층 폴더로 나눈 패키지로 만들고 APIRouter로 창구별 주소를 묶습니다.
2. 상담 요청과 응답의 스키마를 정하고, async 엔드포인트에서 같은 파드의 vLLM에 요청합니다.
3. 여러 요청을 동시에 보내 vLLM이 함께 처리하는 것을 확인합니다.
4. vLLM의 토큰 조각을 SSE로 중계하고, 생각 조각과 답 조각을 나눠 받습니다.

</div>

## 1. 실습 준비와 vLLM 서버 실행

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">vLLM 서버는 모델을 GPU에 올리는 데 몇 분이 걸립니다. 그래서 <strong style="font-size:1.13em;color:#1f3a5f;">키 생성과 vLLM 서버 실행을 먼저 해 두고</strong>, 서버가 뜨는 동안 노트북 준비, FastAPI 서버 실행(2절), 주소 목록 확인(3-2)까지 진행합니다.</p>
</div>

### 1-1. 원격 노트북과 터미널의 역할

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">VS Code Remote SSH로 RunPod에 접속한 원격 창에서 실습합니다. 원격 실습 저장소(일차별 폴더와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">pyproject.toml</code>이 있는 폴더)를 열고, 노트북 오른쪽 위 <strong style="font-size:1.13em;color:#1f3a5f;">Select Kernel → Python Environments... → .venv</strong>를 선택합니다.</p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">작업</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">실행할 곳</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">역할</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">키 생성, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">vllm serve</code> 실행과 종료(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code>)</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">원격 터미널 1</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">모델을 GPU에 올리고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">127.0.0.1:8000</code>에서 요청을 받습니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">uvicorn</code>으로 FastAPI 서버 실행과 종료</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">원격 터미널 2</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">서빙 API를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">127.0.0.1:8080</code>에서 띄우고, 받은 요청을 vLLM에 보냅니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">요청 보내기, 응답과 조각 도착 시각 확인</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">원격 노트북</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">FastAPI 서버에 httpx로 요청하는 클라이언트입니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">서버 코드(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving_app/</code> 아래 .py 파일) 읽기, 함께 따라하기 빈칸 채우기</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">VS Code 편집기</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">파일을 저장하면 터미널 2의 FastAPI 서버가 다시 시작합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">API 문서 화면(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/docs</code>) 열기</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">내 PC 브라우저</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">VS Code 포트 전달로 파드의 8080 포트를 엽니다.</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">아래 Python 셀은 모두 원격 노트북에서 실행합니다.</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">RunPod 터미널</code> 라벨이 붙은 명령은 같은 원격 창의 <strong style="font-size:1.13em;color:#1f3a5f;">Terminal → New Terminal</strong>에서 실행합니다. 터미널 블록 앞부분의 준비 명령이 터미널 1은 일차 폴더(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">day77_FastAPI_모델서빙API</code>)로, 터미널 2는 이 교안 폴더로 이동시킵니다.</p>
</div>

### 1-2. vLLM 키 생성

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">먼저 원격 터미널 1에서 vLLM 키를 생성합니다.</strong> vLLM 키는 vLLM 서버가 요청마다 확인하는 비밀 문자열로, 서빙 벤치마킹 단원의 서버 인증 키와 같습니다. 그 단원과 같은 명령으로 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">enkoa-vllm-</code> 뒤에 무작위 문자열을 붙여 키를 만들고, 원격 저장소의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_API_KEY</code>로 저장합니다.</p>
</div>

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · RunPod 터미널 1</strong></div>

```bash
# 원격 저장소(실습자료) 맨 위 폴더를 찾아 그 안의 폴더로 이동합니다(저장소 안 어느 폴더에서 실행해도 됩니다)
cd "$(git rev-parse --show-toplevel)/day77_FastAPI_모델서빙API"
pwd   # 끝이 day77_FastAPI_모델서빙API이면 맞습니다
mkdir -p output   # 서버 로그를 남길 폴더입니다
# 서빙 벤치마킹 단원과 같은 방식으로 vLLM 키를 생성해 원격 저장소의 .env에 저장합니다(키 값은 화면에 보이지 않습니다)
VLLM_API_KEY="$(uv run python -c 'import secrets; print("enkoa-vllm-" + secrets.token_urlsafe(32))')" &&
export VLLM_API_KEY &&
uv run python -c 'import os; from dotenv import set_key; set_key("../.env", "VLLM_API_KEY", os.environ["VLLM_API_KEY"])' &&
grep -c '^VLLM_API_KEY=' ../.env   # 1이 나오면 저장된 것입니다
```

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">명령어 설정</strong></p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">설정</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">의미</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">secrets.token_urlsafe(32)</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">무작위 문자열을 만들어 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">enkoa-vllm-</code> 뒤에 붙입니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">export VLLM_API_KEY</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">이 터미널의 환경 변수에 키를 넣습니다. 바로 아래 vLLM 서버 명령의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--api-key &quot;&#36;VLLM_API_KEY&quot;</code>가 이 값을 씁니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">set_key(&quot;../.env&quot;, &quot;VLLM_API_KEY&quot;, ...)</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">원격 저장소(실습자료) 폴더의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>에 키를 저장합니다. FastAPI 서버가 이 값을 읽어 vLLM에 요청할 때 붙입니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">grep -c '^VLLM_API_KEY=' ../.env</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_API_KEY=</code>로 시작하는 줄의 수를 셉니다. 1이 나오면 저장된 것이고, 키 값은 화면에 보이지 않습니다.</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;">터미널 1은 닫지 않고 바로 아래 1-3의 서버 실행에 이어 씁니다. 키를 다시 생성했다면 vLLM 서버와 FastAPI 서버를 모두 다시 시작합니다.</p>
</div>

### 1-3. RunPod에서 vLLM 서버 실행

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">서빙 벤치마킹 단원과 같은 모델 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Qwen/Qwen3.5-4B</code>과 같은 옵션으로 vLLM 서버를 띄웁니다. 키를 생성한 <strong style="font-size:1.13em;color:#1f3a5f;">RunPod 터미널 1</strong>에서 실행합니다.</p>
</div>

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · RunPod 터미널 1</strong></div>

```bash
uv run vllm serve Qwen/Qwen3.5-4B \
  --served-model-name qwen3.5-4b \
  --max-model-len 8192 \
  --language-model-only \
  --reasoning-parser qwen3 \
  --default-chat-template-kwargs '{"enable_thinking": false}' \
  --enable-prompt-tokens-details \
  --enable-prefix-caching \
  --host 0.0.0.0 \
  --port 8000 \
  --api-key "$VLLM_API_KEY" 2>&1 | tee output/qwen3.5-4b_server.log
```

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">명령어 설정</strong></p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">설정</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">의미</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--reasoning-parser qwen3</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">생각(reasoning)은 모델이 답을 쓰기 전에 먼저 만드는 풀이 글입니다(오픈소스 LLM 로컬 실행 단원의 추론 모드). 이 옵션은 생각과 답을 응답의 다른 항목에 나눠 담고, 4절과 6절에서 두 항목을 따로 받습니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--default-chat-template-kwargs '{&quot;enable_thinking&quot;: false}'</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">생각을 기본으로 끕니다. 요청 하나에서만 생각을 켜는 방법은 4-2에 있습니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--host 0.0.0.0</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--port 8000</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">모든 IPv4 네트워크 인터페이스에서 요청을 받습니다. 같은 파드의 FastAPI는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">127.0.0.1:8000</code>으로 요청합니다. 외부에서는 공개한 포트의 proxy 주소를 사용합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--api-key &quot;&#36;VLLM_API_KEY&quot;</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">1-2에서 만든 키가 없는 요청을 거절합니다.</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;">나머지 옵션(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--served-model-name</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--max-model-len</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--language-model-only</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--enable-prompt-tokens-details</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--enable-prefix-caching</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">tee</code>)은 서빙 벤치마킹 단원 교안 01 1-3과 같습니다.</p>
</div>

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">로그에 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Application startup complete</code></strong>가 나오면 vLLM 서버가 요청을 받을 수 있습니다. 서버 출력은 RunPod의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/qwen3.5-4b_server.log</code>에도 남습니다. <strong style="font-size:1.13em;color:#1f3a5f;">RunPod 터미널 1</strong>은 그대로 둡니다.</p>
<p style="margin:6px 0;">첫 실행은 모델 다운로드와 GPU 계산 준비에 몇 분이 걸릴 수 있습니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">서버가 뜨는 동안 1-4의 노트북 준비와 2절, 3-1, 3-2를 진행합니다.</strong> vLLM이 필요한 첫 셀은 3-3입니다. 서버 로그 첫머리의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">non-default args</code> 줄에 vLLM 키가 찍히므로, 서버 로그나 터미널 1 화면은 다른 사람에게 공유하지 않습니다.</p>
</div>

### 1-4. 노트북 준비와 노무 질문

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">준비 셀은 자료 경로와 FastAPI 서버 주소를 정합니다. 노트북은 FastAPI 서버에만 요청하고, vLLM 키는 FastAPI 서버가 vLLM에 요청할 때 붙입니다. 그래서 노트북은 키를 읽지 않습니다.</p>
</div>

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · RunPod 원격 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">VS Code 원격 창의 노트북. 내 PC가 아니라 RunPod에서 실행됩니다</span></div>

In [ ]:
import json
from pathlib import Path

import httpx
import pandas as pd

lesson_dir = Path(".").resolve()  # 이 노트북의 단원 폴더(정답 노트북은 "..")
day_dir = lesson_dir.parent                # data, images 가 있는 일차 폴더
data_dir = day_dir / "data"

SERVING_URL = "http://127.0.0.1:8080"   # 터미널 2에서 띄운 FastAPI 서빙 API

# 테이블의 긴 글(질문, 답)이 잘리지 않게 칸의 글자 수 제한을 풉니다
pd.set_option("display.max_colwidth", None)

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">노무 상담 질문은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">data/본실습/labor_questions.jsonl</code>에 있습니다(질문 169개). 시스템 메시지(지침 문장과 근로기준법 참고 조문)는 FastAPI 서버가 붙이므로, 노트북은 질문만 읽습니다.</p>
</div>

In [ ]:
def load_jsonl(path):
    """JSONL 파일을 한 줄에 딕셔너리 하나씩 읽어 리스트로 돌려줍니다(서빙 벤치마킹 단원과 같은 함수)."""
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows



# 노무 상담 질문(id, 질문, 출처 주제)을 읽어 앞 3개를 봅니다
labor_questions = load_jsonl(data_dir / "본실습" / "labor_questions.jsonl")
labor_question_table = pd.DataFrame(labor_questions)
display(labor_question_table.head(3))

## 2. 서빙 API 패키지와 FastAPI 서버 실행

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">교안 00에서는 앱 하나를 .py 파일 하나에 두었습니다. 서빙 API는 주소, vLLM 연결, 양식, 설정처럼 하는 일에 따라 폴더를 나눈 패키지로 만듭니다. 이 교안에서 <strong style="font-size:1.13em;color:#1f3a5f;">창구</strong>는 상담 분야(노무, 전월세)마다 따로 연 주소 묶음을 말합니다.</p>
</div>

### 2-1. serving_app 패키지의 계층 구조

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">서버 코드는 이 노트북과 같은 교안 폴더의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving_app/</code> 패키지(파이썬 파일을 모은 폴더)에 있습니다. 교안 폴더 안 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">정답/serving_app/</code>은 함께 따라하기의 빈칸까지 채운 완성본입니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">계층 구조(layered architecture):</strong> 코드를 하는 일에 따라 층(폴더)으로 나누고, 요청이 위층에서 아래층으로 차례로 내려가게 하는 구조입니다. 이렇게 나누면 한 층을 바꿀 때 다른 층은 고치지 않아도 됩니다. 예를 들어 vLLM 대신 다른 모델 서버를 쓰면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">services/</code>만, 주소를 바꾸면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">routers/</code>만 고칩니다.</p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">폴더, 파일</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">계층과 하는 일</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">보는 절</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">main.py</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">앱 조립: 앱을 만들고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">routers/</code>의 주소 묶음 셋을 붙이며, 서버를 끌 때 vLLM 연결을 닫습니다(lifespan).</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">2절, 3절</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">routers/</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">표현 계층: 주소마다 요청을 받아 서비스 계층에 넘기고, 결과를 응답 양식으로 돌려줍니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">system_router.py</code>(상태 확인), <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">labor_router.py</code>(노무 상담), <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">lease_router.py</code>(전월세 상담, 빈칸)</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">3~7절</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">services/</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">서비스 계층: <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">backend.py</code>는 vLLM에 요청하고 응답을 응답 양식으로 바꾸며, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">streaming.py</code>는 vLLM이 보낸 조각을 서버 전송 이벤트(SSE, 6절)로 바꿉니다. 주소(HTTP)는 모릅니다.</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">4절, 6절</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">schemas/chat.py</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">요청과 응답의 양식(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ChatRequest</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ChatResponse</code>)입니다. 라우터와 서비스가 같은 양식으로 주고받습니다.</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">4절</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">core/config.py</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">설정: 환경 변수와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>에서 vLLM 주소, 키, 모델 이름, 시간 제한을 읽습니다.</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">4절</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">resources/</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">시스템 메시지에 붙일 참고 조문(JSON)입니다.</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">4절</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;">폴더마다 있는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">__init__.py</code>는 그 폴더를 파이썬 패키지로 만들고, 첫 줄에 그 계층이 하는 일을 적어 둡니다.</p>
<p style="margin:6px 0;">요청 하나는 이 계층을 차례로 지납니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">POST /v1/labor/chat</code> 요청은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">main.py</code>의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">app</code>이 받아 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">routers/labor_router.py</code>의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">labor_chat</code>에 넘기고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">labor_chat</code>은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">services/backend.py</code>의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ask_vllm</code>으로 vLLM(127.0.0.1:8000)에 요청합니다. vLLM의 응답은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">services/backend.py</code>의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">to_chat_response</code>가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">schemas/chat.py</code>의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ChatResponse</code> 양식으로 바꾸고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">labor_chat</code>이 이 양식으로 노트북에 돌려줍니다.</p>
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">main.py</code>는 앱 객체 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">app</code>을 만들고 주소 묶음 셋을 붙입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">title</code>과 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">description</code>은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/docs</code> 화면 맨 위에 보입니다.</p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">상대 import:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">from .routers.labor_router import labor_router</code>의 점 하나는 이 파일이 있는 폴더(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving_app/</code>)입니다. 계층 폴더 안의 파일은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">from ..services.backend import ask_vllm</code>처럼 점 두 개로 한 단계 위 폴더(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving_app/</code>)에서 다른 계층을 찾습니다.</li>
</ul>
</div>

```python
# main.py
"""교안 01 서빙 API. 실행(교안 01 폴더): uv run uvicorn serving_app.main:app --host 0.0.0.0 --port 8080 --reload"""
from contextlib import asynccontextmanager

from fastapi import FastAPI

from .routers.labor_router import labor_router
from .routers.lease_router import lease_router
from .routers.system_router import system_router
from .services.backend import vllm_client


@asynccontextmanager
async def lifespan(app: FastAPI):
    # vllm_client 는 여러 라우터 파일이 함께 쓰므로 app.state 대신 backend.py 에서 만든다(만들 때는 연결을 열지 않는다)
    try:
        yield  # 준비가 끝나면 FastAPI가 요청을 처리한다.
    finally:
        await vllm_client.close()  # 종료 때 요청 처리에 쓴 vLLM 연결풀을 닫는다.


# title, description 은 /docs 문서 화면 맨 위에 보인다
app = FastAPI(
    title="노무, 전월세 상담 API",
    description="같은 파드의 vLLM(qwen3.5-4b)에 질문을 보내 상담 답을 돌려줍니다.",
    version="1.0.0",
    lifespan=lifespan,
)

# 기능별 주소 묶음을 앱에 붙인다. 묶음의 prefix 가 주소 앞부분이 되고, 붙이지 않은 묶음의 주소는 404 다
app.include_router(system_router)
app.include_router(labor_router)
app.include_router(lease_router)
```

### 2-2. RunPod 터미널 2에서 FastAPI 서버 실행

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">FastAPI 서버는 시작할 때 vLLM에 요청하지 않으므로, 터미널 1의 vLLM 서버가 다 뜨기 전에 띄워도 됩니다. 원격 창에서 새 터미널(<strong style="font-size:1.13em;color:#1f3a5f;">RunPod 터미널 2</strong>)을 열고 아래 블록을 실행합니다. 교안 00에서 쓴 uvicorn 명령과 같고, 앱 경로만 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving_app.main:app</code>입니다.</p>
</div>

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · RunPod 터미널 2</strong></div>

```bash
# 새 원격 터미널이면 이 블록부터 실행합니다
# 원격 저장소(실습자료) 맨 위 폴더를 찾아 그 안의 폴더로 이동합니다(저장소 안 어느 폴더에서 실행해도 됩니다)
cd "$(git rev-parse --show-toplevel)/day77_FastAPI_모델서빙API/교안_01_FastAPI로_vLLM_서빙_API_만들기"
pwd   # 끝이 교안_01_FastAPI로_vLLM_서빙_API_만들기이면 맞습니다

# FastAPI 서버를 띄웁니다(멈출 때까지 이 터미널을 씁니다)
# serving_app/main.py 의 app 을 8080 포트로 띄웁니다(8000 은 vLLM 이 씁니다)
uv run uvicorn serving_app.main:app --host 0.0.0.0 --port 8080 --reload
```

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">실행 폴더:</strong> uvicorn은 명령을 실행한 폴더에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving_app</code> 패키지를 찾습니다. 학생 노트북의 명령은 교안 폴더로 이동해 빈칸이 있는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving_app</code>을, 정답 노트북의 명령은 교안 폴더의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">정답/</code>으로 이동해 완성본 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">정답/serving_app</code>을 띄웁니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">vLLM 키:</strong> FastAPI 서버는 시작할 때 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">core/config.py</code>가 실습자료 폴더의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_API_KEY</code>를 읽습니다. 터미널 2에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ValidationError</code>와 함께 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">vllm_api_key</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Field required</code>가 나오면 1-2의 키 생성을 마치지 않은 것입니다.</li>
</ul>
</div>

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">터미널 2에 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Application startup complete</code></strong>와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Uvicorn running on http://0.0.0.0:8080</code>가 나오면 FastAPI 서버가 요청을 받을 수 있습니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--reload</code>를 붙였으므로 서버 코드(.py)를 저장하면 서버가 저절로 다시 시작합니다.</p>
</div>

### 2-3. 서버 상태 확인(/healthz)

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">노트북은 교안 00과 같이 httpx로 FastAPI 서버에 요청합니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/healthz</code>는 서빙 API가 요청을 받을 수 있는지만 확인하는 주소라 vLLM을 부르지 않습니다.</p>
</div>

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · RunPod 원격 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">VS Code 원격 창의 노트북. 내 PC가 아니라 RunPod에서 실행됩니다</span></div>

In [ ]:
# 서빙 API의 상태 확인 주소에 GET 요청을 보냅니다
health_response = httpx.get(f"{SERVING_URL}/healthz")

# 상태 코드와 응답 본문(JSON)을 함께 봅니다
health_check = {"상태 코드": health_response.status_code, "본문": health_response.json()}
display(health_check)

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">상태 코드 200, 본문 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">{'status': 'ok'}</code>:</strong> 터미널 2의 FastAPI 서버가 요청을 받고 있습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">ConnectError</code>가 나면:</strong> 터미널 2의 FastAPI 서버가 떠 있지 않은 것입니다. 2-2의 명령과 터미널 2의 오류 줄을 확인합니다.</li>
</ul>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">API 문서 화면:</strong> 교안 00의 '자동 문서(/docs)' 절과 같이 PORTS 탭에서 8080 포트를 전달하고, 내 PC 브라우저에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">http://localhost:8080/docs</code>를 엽니다. 화면의 묶음 이름(상태 확인, 노무 상담, 전월세 상담)은 3절의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">tags</code>입니다.</p>
</div>

## 3. 라우팅

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">라우팅(routing)</strong>은 요청의 주소와 방식(GET, POST)을 보고 처리할 함수를 고르는 일입니다. 엔드포인트를 만드는 방법은 교안 00의 '첫 엔드포인트와 매개변수' 절과 같습니다. 다만 서빙 API는 엔드포인트를 앱이 아니라 창구별 묶음에 붙입니다.</p>
</div>

### 3-1. APIRouter로 주소 묶기

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">APIRouter</code></strong>는 엔드포인트 여러 개를 기능별 묶음 하나로 모으는 FastAPI 객체입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">routers/</code> 폴더에 창구마다 파일 하나, 묶음 하나를 두면, 창구를 더하거나 뺄 때 그 파일과 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">main.py</code>의 한 줄만 바꿉니다.</p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">함수 몸통:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">labor_chat</code> 안의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ask_vllm</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">to_chat_response</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">settings.max_tokens_think_labor</code>는 4절에서 봅니다. 여기서는 데코레이터와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">prefix</code>가 주소를 정하는 방식만 봅니다.</li>
</ul>
</div>

```python
# routers/labor_router.py
# prefix 는 묶음 안 모든 주소 앞에 붙고(/chat 은 /v1/labor/chat), tags 는 /docs 화면에서 묶어 보여 줄 이름이다
labor_router = APIRouter(prefix="/v1/labor", tags=["노무 상담"])


# 교안 00 의 @app.post 와 같은 데코레이터지만, 함수를 앱이 아니라 묶음에 등록한다
@labor_router.post("/chat", response_model=ChatResponse)
async def labor_chat(request: ChatRequest):
    """노무 질문 하나에 대한 답 전체를 한 번에 돌려준다."""
    # vLLM 의 답을 기다리는 동안(await) 서버는 다른 요청을 받는다
    # 시스템 메시지가 길어 생각 상한은 노무 창구 값(settings.max_tokens_think_labor)을 넘긴다
    # 입력 토큰 수와 생성 상한의 합이 vLLM 의 --max-model-len(8192)을 넘으면 vLLM 이 거절한다
    completion = await ask_vllm(LABOR_SYSTEM_PROMPT, request.question, request.think, max_tokens_think=settings.max_tokens_think_labor)
    return to_chat_response(completion)
```

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">상태 확인 묶음 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">system_router</code>는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">prefix</code> 없이 주소 전체(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/healthz</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1/models</code>)를 적었습니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">prefix</code>를 쓸지는 묶음마다 정할 수 있습니다.</p>
</div>

```python
# routers/system_router.py
system_router = APIRouter(tags=["상태 확인"])


@system_router.get("/healthz")
async def healthz():
    """서빙 API 가 요청을 받을 수 있으면 {"status": "ok"} 를 돌려준다(vLLM 은 부르지 않는다)."""
    return {"status": "ok"}


@system_router.get("/v1/models")
async def list_models():
    """vLLM 서버가 서빙 중인 모델 이름 목록을 돌려준다."""
    models = await vllm_client.models.list()
    names = []
    for model in models.data:
        names.append(model.id)
    return {"models": names}
```

### 3-2. 앱에 붙은 주소 목록

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/openapi.json</code>은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/docs</code> 화면이 읽는 API 문서입니다. 이 문서에서 앱에 붙은 엔드포인트를 꺼내 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">prefix</code>가 붙은 실제 주소를 확인합니다.</p>
</div>

In [ ]:
# FastAPI가 만든 API 문서(openapi.json)를 받습니다
response = httpx.get(f"{SERVING_URL}/openapi.json")
response.raise_for_status()
openapi = response.json()

# 주소마다 방식(GET, POST)과 묶음 이름(tags)을 한 행으로 모읍니다. summary는 FastAPI가 함수 이름으로 만든 제목입니다
route_rows = []
for path, operations in openapi["paths"].items():
    for method, operation in operations.items():
        route_rows.append({"주소": path, "방식": method.upper(), "tags": operation["tags"], "summary": operation["summary"]})
display(pd.DataFrame(route_rows))

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">주소</code> 열:</strong> 노무 창구의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/chat</code>과 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/chat/stream</code>이 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">prefix</code>가 붙은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1/labor/chat</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1/labor/chat/stream</code>으로 보입니다. 전월세 창구도 같은 모양입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">tags</code> 열:</strong> 같은 묶음의 엔드포인트가 같은 이름을 가지며, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/docs</code> 화면의 묶음과 같습니다.</li>
</ul>
</div>

### 3-3. vLLM 모델 목록(/v1/models)

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">터미널 1에 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Application startup complete</code>가 나온 뒤 실행합니다.</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1/models</code>는 FastAPI 서버가 vLLM에 모델 목록을 물어 이름만 돌려주는 주소입니다. 이 요청이 성공하면 FastAPI 서버가 설정한 vLLM 주소와 키로 vLLM에 제대로 닿는다는 뜻입니다.</p>
</div>

In [ ]:
# FastAPI 서버를 거쳐 vLLM의 모델 목록을 받습니다(노트북은 vLLM 키를 보내지 않습니다)
response = httpx.get(f"{SERVING_URL}/v1/models", timeout=60)
response.raise_for_status()
models_reply = response.json()
display(models_reply)

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">models</code>의 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">qwen3.5-4b</code>:</strong> vLLM 서버의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--served-model-name</code>입니다. vLLM 키는 FastAPI 서버가 vLLM에 요청할 때 붙였습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">500(Internal Server Error)이 오면:</strong> 터미널 1의 vLLM 서버가 떠 있는지 확인합니다. 이 교안의 앱은 vLLM 연결 오류를 따로 처리하지 않아 500으로 돌려줍니다.</li>
</ul>
</div>

## 4. vLLM 백엔드 연동

### 4-1. 상담 요청과 응답의 스키마

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">BaseModel</code>과 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Field</code>로 양식을 정하는 방법, 양식이 틀린 요청이 422로 돌아오는 것은 교안 00의 '요청 본문과 스키마' 절과 같습니다. 여기서는 상담 창구가 주고받는 항목을 정합니다.</p>
</div>

```python
# schemas/chat.py
class ChatRequest(BaseModel):
    """상담 질문 하나. 시스템 메시지와 생성 상한은 항목에 없어 사용자가 바꿀 수 없다."""

    # 빈 질문과 지나치게 긴 질문은 vLLM 에 보내기 전에 422 로 돌려보낸다
    question: str = Field(min_length=2, max_length=500, description="사용자의 질문")
    think: bool = Field(default=False, description="True 면 모델의 생각(reasoning)을 켜고 답과 따로 돌려준다")


class Usage(BaseModel):
    """요청 하나가 쓴 토큰 수."""

    prompt_tokens: int = Field(description="입력 토큰 수(시스템 메시지 포함)")
    cached_tokens: int = Field(description="캐시 적중 토큰: 입력 토큰 가운데 캐시에서 다시 쓴 토큰 수")
    completion_tokens: int = Field(description="출력 토큰 수(생각 포함)")
    total_tokens: int = Field(description="입력과 출력의 합")


class ChatResponse(BaseModel):
    """상담 답변 하나. finish_reason, model, usage 는 vLLM 응답의 값을 그대로 옮겨 담는다."""

    # 생각이 생성 상한을 다 쓰면 답이 비어 None 이 될 수 있다(finish_reason 이 length)
    answer: str | None = Field(description="모델의 답")
    reasoning: str | None = Field(default=None, description="모델의 생각. think 가 False 면 None")
    finish_reason: str = Field(description="생성이 끝난 이유. stop 은 답을 마침, length 는 생성 상한에 닿음")
    model: str = Field(description="실제로 답한 모델 이름")
    usage: Usage = Field(description="토큰 사용량")
```

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">422와 vLLM:</strong> 양식 검사는 엔드포인트 함수를 부르기 전에 하므로, 양식이 틀린 요청은 vLLM에 닿지 않고 GPU를 쓰지 않습니다.</li>
</ul>
</div>

### 4-2. vLLM에 요청하는 코드

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">vLLM 서버는 OpenAI 호환 API(vLLM 추론 엔진 단원, openai 라이브러리로 요청할 수 있는 API)를 제공하므로, FastAPI 서버는 openai 라이브러리의 클라이언트에 vLLM 주소를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">base_url</code>로 주어 요청합니다. 이때 동기 클라이언트 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">OpenAI</code> 대신 비동기 클라이언트 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">AsyncOpenAI</code></strong>를 씁니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">async def</code> 엔드포인트가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">await</code>로 vLLM의 답을 기다리는 동안 다른 요청을 받아야 하기 때문입니다. 동기 클라이언트를 쓰면 교안 00의 'def와 async def' 절에서 본 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">time.sleep</code>처럼 기다리는 동안 다른 요청까지 멈춥니다.</p>
<p style="margin:6px 0;">주소, 키, 모델 이름, 시간 제한은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">core/config.py</code>에서 읽습니다. 이 파일은 <strong style="font-size:1.13em;color:#1f3a5f;">pydantic-settings</strong>의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">BaseSettings</code>로 설정을 읽습니다. FastAPI 공식 문서(Settings and Environment Variables)도 이 방법을 권합니다.</p>
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Settings</code> 클래스에 필드 이름과 타입, 기본값을 적어 두면, 필드 이름을 대문자로 쓴 환경 변수(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">vllm_base_url</code>이면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_BASE_URL</code>)나 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>의 값을 그 타입으로 바꿔 채워 줍니다. 셸에서 준 환경 변수가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>의 값보다 먼저 쓰이고, 둘 다 없으면 기본값을 씁니다. 기본값이 없는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">vllm_api_key</code>가 아예 없거나 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_TIMEOUT_SECONDS</code>에 숫자가 아닌 값이 들어 있으면, 요청을 받기 전에 서버가 뜨면서 어떤 이름이 문제인지 알려 줍니다.</p>
<p style="margin:6px 0;">생성 상한과 샘플링 값도 이 클래스에 둡니다. 모델이나 vLLM 서버의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--max-model-len</code>을 바꾸면 이 값도 함께 바뀌어야 하므로, 코드를 고치지 않고 환경 변수로 바꿀 수 있게 한 것입니다. 기본값은 이 교안의 vLLM 설정(Qwen3.5-4B, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--max-model-len 8192</code>)에 맞춘 값이고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Field(gt=0)</code>처럼 범위를 적어 두면 0 이하 같은 잘못된 값을 서버가 뜰 때 막습니다. 서비스 키 파일, 키 DB, 관리자 키, OpenAI 키는 교안 02, 교안 03, 부록 01에서 씁니다.</p>
</div>

```python
# core/config.py
class Settings(BaseSettings):
    """환경 변수에서 읽는 설정. 필드 이름을 대문자로 쓴 것이 환경 변수 이름이다(vllm_base_url -> VLLM_BASE_URL)."""

    # 이 클래스는 이름, 타입, 기본값을 적어 둔 설계도다. 클래스를 정의할 때는 아무 값도 읽지 않는다.
    # 맨 아래 settings = Settings() 가 실행될 때 필드마다 환경 변수 -> .env -> 기본값 순서로 찾아 처음 찾은 값을 넣는다.
    # 그래서 "= 값" 은 다른 곳에 값이 없을 때만 쓰는 기본값이고, 줄의 순서와 상관없이 환경 변수가 먼저 쓰인다.
    # 환경 변수 이름은 대소문자를 구분하지 않는다(case_sensitive 의 기본값이 False). 소문자 필드 vllm_base_url 을 VLLM_BASE_URL 이 채운다.

    # 서버를 실행한 폴더에서 위로 올라가며 찾은 .env(실습자료/.env)도 읽는다. .env 의 다른 키는 무시한다(extra="ignore").
    # 값이 틀려 서버가 멈출 때 오류 메시지에 입력값을 찍지 않는다(hide_input_in_errors). 키가 로그에 남지 않게 한다
    model_config = SettingsConfigDict(
        env_file=find_dotenv(usecwd=True) or None,
        extra="ignore",
        hide_input_in_errors=True,
    )

    # vLLM 서버 주소. 같은 파드에서는 127.0.0.1:8000, 배포 단원에서는 다른 파드의 주소로 바꾼다
    vllm_base_url: str = "http://127.0.0.1:8000/v1"

    # vLLM 서버를 --api-key 로 띄울 때 쓴 키. 기본값이 없으므로 반드시 정해야 한다(사용자에게는 알려 주지 않는다)
    vllm_api_key: str

    # vLLM 의 --served-model-name
    served_name: str = "qwen3.5-4b"

    # vLLM 응답을 기다리는 최대 시간(초). 넘기면 시간 초과 오류가 난다
    vllm_timeout_seconds: float = 120

    # 생각을 끈 요청의 생성 상한(서빙 벤치마킹 단원과 같은 값). Field(gt=0) 는 0 이하의 값을 서버가 뜰 때 막는다
    max_tokens: int = Field(256, gt=0)

    # 생각을 켠 요청의 생성 상한. 생각도 출력 토큰을 쓰고, 생각이 상한을 다 쓰면 답이 비므로 더 크게 둔다
    # 입력 토큰 수와 이 상한의 합이 vLLM 의 --max-model-len(8192)을 넘으면 vLLM 이 요청을 거절(400)하므로,
    # 전월세 500자 질문의 가장 긴 입력 3,716토큰(글자마다 최대 4토큰)이 들어가도록 8192 - 3716 = 4476 보다 작게 둔다.
    # --max-model-len 이나 모델을 바꾸면 이 값도 함께 다시 계산한다
    max_tokens_think: int = Field(4400, gt=0)

    # 시스템 메시지가 긴 노무 창구(입력 약 5,000토큰)의 생각 상한. max_tokens_think 를 주면 합이 8192 를 넘으므로 줄여 둔다
    max_tokens_think_labor: int = Field(2048, gt=0)

    # 생각을 켠 요청의 샘플링 값(Qwen3.5 모델 카드의 생각 모드 권장값). 모델을 바꾸면 그 모델의 권장값으로 바꾼다
    # temperature=0 이면 이 모델은 생각을 되풀이하다 상한에 닿는다
    think_temperature: float = Field(1.0, ge=0)
    think_top_p: float = Field(0.95, gt=0, le=1)
    think_top_k: int = Field(20, gt=0)
    think_presence_penalty: float = Field(1.5, ge=-2, le=2)

    # 서비스 키 목록 파일. 배포할 때는 이미지에 넣지 않고 이 환경 변수로 다른 경로를 준다
    service_keys_file: Path = RESOURCE_DIR / "service_keys.json"

    # 서비스 키와 사용량을 저장할 SQLite 파일. 서버를 실행한 폴더 기준 상대 경로이고, 배포할 때는 환경 변수로 바꾼다
    service_db_path: Path = Path("output/service.db")

    # 관리자 키. 관리자 주소를 여는 앱만 쓰므로 기본값은 None 이고, 그 앱이 없으면 멈춘다
    admin_api_key: str | None = None

    # 예비 모델(OpenAI)의 키. 예비 모델을 쓰는 앱만 쓰므로 기본값은 None 이고, 그 앱이 없으면 멈춘다
    openai_api_key: str | None = None


# 여기서 값을 읽는다. 서버가 뜰 때 한 번만 읽으므로, 값을 바꾸려면 서버를 다시 띄운다. 모든 모듈이 이 settings 를 같이 쓴다
settings = Settings()
```

```python
# services/backend.py
# 비동기 클라이언트는 이 파일을 불러올 때 하나만 만들어 모든 요청이 함께 쓴다(요청마다 만들면 연결 준비를 반복한다)
# 만들 때는 연결을 열지 않는다. 첫 요청 때 연 연결을 연결풀에 두고 재사용하다가, 앱 조립 파일의 lifespan 이 종료 때 닫는다
# max_retries=0: 실패를 숨기지 않고 바로 오류로 받는다(다시 시도할지는 서빙 API 가 정한다)
vllm_client = AsyncOpenAI(
    base_url=settings.vllm_base_url,
    api_key=settings.vllm_api_key,
    timeout=settings.vllm_timeout_seconds,  # vLLM 응답을 기다리는 최대 시간(VLLM_TIMEOUT_SECONDS, 기본 120초)
    max_retries=0,
)


def build_request(system_prompt, question, think, max_tokens_think=settings.max_tokens_think):
    """vLLM chat completions 요청에 넣을 인자를 딕셔너리로 돌려준다."""
    # 생각을 끈 요청: 생성 상한 settings.max_tokens, temperature=0(같은 질문에 같은 답)
    request = {
        "model": settings.served_name,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question},
        ],
        "max_tokens": settings.max_tokens,
        "temperature": 0,
        # vLLM 전용 인자: 이 요청에서만 Qwen3.5 의 생각을 켜거나 끈다(서버 기본값은 꺼짐)
        "extra_body": {"chat_template_kwargs": {"enable_thinking": think}},
    }

    # 생각을 켠 요청: 생각도 출력 토큰을 쓰므로 생성 상한을 키우고(시스템 메시지가 긴 창구는 더 작은 값을 넘긴다),
    # 샘플링은 모델 카드 권장값으로 바꾼다(temperature=0 이면 같은 생각을 되풀이하다 상한에 닿는다. 대신 답이 실행마다 달라진다)
    if think:
        request["max_tokens"] = max_tokens_think
        request["temperature"] = settings.think_temperature
        request["top_p"] = settings.think_top_p
        request["presence_penalty"] = settings.think_presence_penalty
        # top_k 는 OpenAI 표준 인자가 아니라 vLLM 전용이므로 extra_body 에 넣는다
        request["extra_body"]["top_k"] = settings.think_top_k
    return request


async def ask_vllm(system_prompt, question, think, max_tokens_think=settings.max_tokens_think):
    """vLLM 에 질문 하나를 보내고 완성된 응답(ChatCompletion)을 돌려준다."""
    return await vllm_client.chat.completions.create(**build_request(system_prompt, question, think, max_tokens_think))


def to_chat_response(completion):
    """vLLM 응답(ChatCompletion)을 서빙 API 의 응답 양식(ChatResponse)으로 바꿔 돌려준다."""
    choice = completion.choices[0]

    # 생각은 OpenAI 표준에 없는 vLLM 전용 항목이라 model_extra 에 들어 있다(생각을 끄면 None)
    reasoning = choice.message.model_extra.get("reasoning")

    return ChatResponse(
        answer=choice.message.content,
        reasoning=reasoning,
        finish_reason=choice.finish_reason,
        model=completion.model,
        usage=Usage(
            prompt_tokens=completion.usage.prompt_tokens,
            # 서버 명령의 --enable-prompt-tokens-details 가 prompt_tokens_details 를 응답에 넣는다
            cached_tokens=completion.usage.prompt_tokens_details.cached_tokens,
            completion_tokens=completion.usage.completion_tokens,
            total_tokens=completion.usage.total_tokens,
        ),
    )
```

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">질문 여러 개가 거의 함께 오면:</strong> 첫 요청이 답을 기다리는 사이 나머지 요청도 vLLM으로 보내집니다. 5절에서 질문 4개로 확인합니다.</li>
</ul>
</div>

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">노무 창구의 시스템 메시지는 서버가 뜰 때 한 번 만듭니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">make_system_prompt</code>는 지침 문장 뒤에 참고 조문을 붙이며, 서빙 벤치마킹 단원과 같은 구성입니다. 3-1에서 본 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">labor_chat</code>은 이 시스템 메시지와 사용자의 질문으로 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ask_vllm</code>을 부르고, 받은 응답을 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">to_chat_response</code>로 바꿔 돌려줍니다.</p>
</div>

```python
# routers/labor_router.py
# 시스템 메시지는 서버가 뜰 때 한 번 만든다. 사용자는 질문만 보내고 시스템 메시지는 바꿀 수 없다
LABOR_SYSTEM_PROMPT = make_system_prompt(
    "너는 노무 상담 챗봇이다. 노동법 조문에 근거해 존댓말로 간결히 답한다.",
    "labor_reference.json",
)
```

### 4-3. 노무 질문 하나 보내기

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">노무 질문의 첫 번째를 일반 응답 주소로 보냅니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">think</code>를 보내지 않았으므로 스키마의 기본값 False가 쓰입니다. 서버를 띄운 뒤 첫 요청은 vLLM이 GPU 계산 준비를 마저 하느라 1분 가까이 걸릴 수 있습니다.</p>
</div>

In [ ]:
# 노무 질문의 첫 번째를 생각을 끈 채 일반 응답 주소로 보냅니다
# httpx의 기본 시간 제한 5초는 모델의 답을 기다리기에 짧고, 첫 요청은 느릴 수 있어 120초로 늘립니다
user_question = labor_questions[0]["question"]
response = httpx.post(f"{SERVING_URL}/v1/labor/chat", json={"question": user_question}, timeout=120)

# 200이 아니면 여기서 오류로 멈춥니다
response.raise_for_status()
labor_reply = response.json()
display(labor_reply)

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">answer</code>와 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">reasoning</code>:</strong> 생각을 껐으므로 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">reasoning</code>이 None이고, 답은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">answer</code>에만 있습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">finish_reason</code>:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">stop</code>은 모델이 답을 스스로 마쳤다는 뜻이고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">length</code>는 생성 상한에 닿아 답이 잘렸다는 뜻입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">length</code>이면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">completion_tokens</code>가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">max_tokens</code>(256)와 같고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">answer</code>가 문장 중간에서 끝납니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">usage</code>의 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">prompt_tokens</code>:</strong> 노트북이 보낸 질문만이 아니라 서버가 붙인 시스템 메시지(참고 조문)까지 센 입력 토큰 수입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">usage</code>의 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">cached_tokens</code>:</strong> 서빙 벤치마킹 단원의 접두어 캐싱에서 본 캐시 적중 토큰입니다. 같은 시스템 메시지로 처음 보낸 요청은 0이고, 다음 요청부터 시스템 메시지 앞부분만큼 커집니다.</li>
</ul>
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">max_tokens</code>(256)는 서빙 벤치마킹 단원과 같은 값으로 둔 실습용 상한입니다. 실제 서비스에서는 잘린 답이 나가지 않게 상한을 올립니다. 다만 올린 만큼 요청 하나가 GPU를 오래 쓰므로, 5절의 동시 처리와 함께 정합니다.</p>
</div>

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">생각을 켠 요청(<code style="color:inherit;background:#f3f6fa;font-size:0.95em;">think=True</code>)은 7절의 전월세 창구로 보냅니다.</strong> 노무 창구는 생각 상한이 2048토큰이라 생각만 하다 답 없이 끝날 수 있습니다.</p>
</div>

## 5. 동시 요청

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">async def</code> 엔드포인트는 vLLM의 답을 기다리는 동안 다른 요청을 받으므로(4-2), 여러 요청이 FastAPI 서버를 지나 vLLM에 함께 도착합니다. vLLM은 함께 들어온 요청을 연속 배치(vLLM 추론 엔진 단원)로 같은 생성 단계에 묶어 계산합니다. 서빙 벤치마킹 단원에서 동시 요청 수를 늘리면 출력 처리량이 커진 것도 이 때문입니다.</p>
<p style="margin:6px 0;">노무 질문 4개를 순차로 한 번, 동시에 한 번 보내 걸린 시간을 비교합니다. 4-3에서 같은 시스템 메시지를 이미 보냈으므로, 두 방식 모두 접두어 캐싱(서빙 벤치마킹 단원 교안 02)으로 시스템 메시지의 계산 결과를 다시 씁니다.</p>
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">httpx.AsyncClient</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">asyncio.Semaphore</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">run_limited</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">asyncio.gather</code>는 교안 00의 'def와 async def' 절과 같습니다. 이번에는 기다리는 작업이 vLLM의 답입니다. 질문 수가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">MAX_CONCURRENCY</code>(4)와 같은 4개라서 네 요청이 함께 출발합니다.</p>
</div>

In [ ]:
import asyncio
import time

# 동시에 보낼 요청 수. 질문 4개를 한꺼번에 보내 순차 4건과 걸린 시간을 비교하려고 4로 둡니다
MAX_CONCURRENCY = 4
semaphore = asyncio.Semaphore(MAX_CONCURRENCY)


async def run_limited(task):
    """세마포어 자리가 날 때까지 기다렸다가 작업 하나를 실행합니다."""
    async with semaphore:
        return await task


async def post_chat(client, path, question):
    """서빙 API의 일반 응답 주소에 질문 하나를 보내고 응답 본문(딕셔너리)을 돌려줍니다."""
    response = await client.post(path, json={"question": question})
    response.raise_for_status()
    return response.json()

In [ ]:
# 노무 질문 앞 4개
demo_questions = [row["question"] for row in labor_questions[:4]]

# 순차: 앞 요청의 답을 받은 뒤 다음 요청을 보냅니다
async with httpx.AsyncClient(base_url=SERVING_URL, timeout=60) as client:
    started = time.perf_counter()
    sequential_replies = []
    for question in demo_questions:
        sequential_replies.append(await post_chat(client, "/v1/labor/chat", question))
    sequential_seconds = time.perf_counter() - started

# 걸린 시간과 요청 4개의 출력 토큰 합을 봅니다
print(f"순차 4건: {sequential_seconds:.2f}초 | 출력 토큰 합:", sum([reply["usage"]["completion_tokens"] for reply in sequential_replies]))

In [ ]:
# 동시: 요청 4개를 함께 보내고 모두 끝날 때까지 기다립니다(결과는 보낸 순서대로 돌아옵니다)
async with httpx.AsyncClient(base_url=SERVING_URL, timeout=60) as client:
    started = time.perf_counter()
    tasks = [run_limited(post_chat(client, "/v1/labor/chat", question)) for question in demo_questions]
    concurrent_replies = await asyncio.gather(*tasks)
    concurrent_seconds = time.perf_counter() - started

# 걸린 시간과 요청 4개의 출력 토큰 합을 봅니다
print(f"동시 4건: {concurrent_seconds:.2f}초 | 출력 토큰 합:", sum([reply["usage"]["completion_tokens"] for reply in concurrent_replies]))

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">걸린 시간:</strong> 순차는 요청 4개의 요청 지연을 더한 만큼 걸리고, 동시는 FastAPI 서버와 vLLM이 네 요청을 함께 처리하므로 그보다 짧아집니다. 함께 계산하면 요청 하나의 지연은 혼자 보낼 때보다 길어질 수 있어, 4분의 1까지 줄지는 않습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">출력 토큰 합:</strong> 두 값이 비슷하면 같은 양의 일을 더 짧은 시간에 끝낸 것입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">temperature=0</code>이어도 함께 계산하면 같은 질문의 답이 조금 달라질 수 있어 두 값이 꼭 같지는 않습니다.</li>
</ul>
</div>

## 6. 스트리밍 중계

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">교안 00의 '스트리밍 응답' 절에서는 만들어지는 대로 조각을 보내는 응답을 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">StreamingResponse</code>와 비동기 제너레이터로 만들었습니다. 서빙 API는 vLLM이 보낸 토큰 조각을 답 전체가 모일 때까지 기다리지 않고, 받는 즉시 같은 순서로 노트북에 다시 보냅니다. 이렇게 받은 조각을 모으지 않고 바로 다음 쪽으로 넘기는 것을 <strong style="font-size:1.13em;color:#1f3a5f;">중계</strong>라고 합니다. 넘길 때 조각의 양식은 서빙 API의 이벤트 양식으로 바꿉니다(6-1). 덕분에 사용자는 요청 지연 전체를 기다리지 않고, 서빙 벤치마킹 단원에서 배운 TTFT(Time To First Token, 첫 토큰 지연)가 지나면 답을 읽기 시작합니다.</p>
<img style="display:block;max-width:100%;height:auto;margin:0;" src="../images/11_stream_relay.png" width="1050" alt="vLLM 서버가 만든 생각 조각, 답 조각, 사용량 조각과 끝 표시가 FastAPI 서빙 API를 거쳐 같은 순서로 노트북에 도착하는 흐름입니다. 요청을 보낸 때부터 첫 조각이 도착할 때까지가 TTFT이고, 생각을 켜면 첫 답 조각은 그보다 늦게 옵니다.">
</div>

### 6-1. SSE와 relay_stream

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">SSE(Server-Sent Events, 서버 전송 이벤트):</strong> 서버가 응답 하나를 열어 둔 채 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">data: 내용</code> 줄을 이어 보내는 웹 표준 형식입니다. 이벤트 사이에는 빈 줄을 두고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">data:</code> 뒤에 무엇을 담을지는 보내는 쪽이 정합니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">생각을 보내는 방법(Qwen3.5):</strong> 생각을 켜면 Qwen3.5의 채팅 템플릿이 프롬프트 끝에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&lt;think&gt;</code>를 열어 두고, 모델은 생각 글을 쓴 뒤 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&lt;/think&gt;</code>로 닫고 답을 씁니다(생각을 끄면 템플릿이 빈 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&lt;think&gt;&lt;/think&gt;</code>를 미리 넣어 바로 답을 씁니다). vLLM은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--reasoning-parser qwen3</code>(1-3)로 이 경계를 찾아, 생각 조각은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">delta</code>의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">reasoning</code> 항목에, 답 조각은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">content</code> 항목에 담아 생각부터 보냅니다. 그래서 받는 쪽은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&lt;think&gt;</code> 표시를 직접 찾지 않습니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">vLLM이 보내는 줄(OpenAI 방식):</strong> 전월세 질문 &quot;1년으로 계약했는데 집주인이 1년 뒤 나가라고 해요.&quot;를 생각을 켜고 보냈을 때의 줄입니다. 생각도 답처럼 만들어지는 대로 조각으로 오고, 생각 조각이 모두 온 뒤에 답 조각이 이어집니다(일반 응답은 생각 전체가 응답의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">reasoning</code> 항목에 한 번에 들어옵니다). 생각은 영어로 나올 때가 많고, 아래 생각 조각은 &quot;사용자 상황: 1년 계약을 했고, 집주인이 1년 뒤 나가라고 한다&quot;라는 뜻입니다(실행에서 나온 긴 생각 글 가운데 한 문장입니다. 실제 조각은 토큰 한두 개씩이지만 읽기 쉽게 몇 낱말씩 묶었고, 앞뒤 줄과 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">id</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">model</code> 같은 항목은 뺐습니다).</p>
</div>

```text
data: {"choices": [{"delta": {"reasoning": "**User's Situation:** Signed a 1-year lease,"}}]}

data: {"choices": [{"delta": {"reasoning": " landlord wants to evict after 1 year."}}]}

data: {"choices": [{"delta": {"content": "주택임대차보호법에 따라"}}]}

data: [DONE]
```

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">서빙 API가 보내는 줄(이 교안의 규칙):</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">relay_stream</code>이 위 조각을 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">type</code>이 붙은 짧은 JSON으로 바꿉니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">reasoning</code> 항목은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">type: reasoning</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">content</code> 항목은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">type: answer</code>가 되고, 마지막의 토큰 사용량은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">type: usage</code> 줄 하나로 보냅니다(6-2). 받는 쪽은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">type</code> 하나로 생각, 답, 사용량을 나눕니다.</p>
</div>

```text
data: {"type": "reasoning", "text": "**User's Situation:** Signed a 1-year lease,"}

data: {"type": "reasoning", "text": " landlord wants to evict after 1 year."}

data: {"type": "answer", "text": "주택임대차보호법에 따라"}

data: [DONE]
```

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">줄의 부분</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">정한 곳</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">data:</code> 줄과 이벤트 사이의 빈 줄</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">SSE 표준</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">choices</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">delta</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">content</code>, 끝 표시 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">[DONE]</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">OpenAI 방식(vLLM이 따름)</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">delta</code> 안의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">reasoning</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">vLLM(OpenAI 표준에 없는 항목)</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">type</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">text</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">이 서빙 API</td>
</tr>
</tbody>
</table>
</div>

```python
# services/backend.py
async def open_vllm_stream(system_prompt, question, think, max_tokens_think=settings.max_tokens_think):
    """vLLM 에 스트리밍 요청을 보내고, 조각을 하나씩 꺼낼 수 있는 스트림을 돌려준다."""
    return await vllm_client.chat.completions.create(
        **build_request(system_prompt, question, think, max_tokens_think),
        stream=True,  # vLLM 이 조각을 만들어지는 대로 하나씩 보낸다
        # 마지막에 토큰 사용량만 담은 조각을 하나 더 받는다
        stream_options={"include_usage": True},
    )
```

```python
# services/streaming.py
def sse_line(event):
    """이벤트 딕셔너리 하나를 SSE 한 줄("data: ..." + 빈 줄)로 바꿔 돌려준다."""
    return "data: " + json.dumps(event, ensure_ascii=False) + "\n\n"


async def relay_stream(stream):
    """vLLM 스트림의 조각을 받는 대로 SSE 이벤트로 바꿔 하나씩 내보낸다(비동기 제너레이터)."""
    async for chunk in stream:
        # 1) 생각 조각과 답 조각: 조각마다 choices 가 하나 들어 있다
        for choice in chunk.choices:
            # 생각은 vLLM 전용 항목이라 model_extra 에 들어 있다
            reasoning = choice.delta.model_extra.get("reasoning")
            if reasoning:
                yield sse_line({"type": "reasoning", "text": reasoning})
            if choice.delta.content:
                yield sse_line({"type": "answer", "text": choice.delta.content})

        # 2) 마지막 조각: choices 가 비어 있고 토큰 사용량만 들어 있다
        if chunk.usage is not None:
            yield sse_line({
                "type": "usage",
                "prompt_tokens": chunk.usage.prompt_tokens,
                "cached_tokens": chunk.usage.prompt_tokens_details.cached_tokens,  # 캐시 적중 토큰
                "completion_tokens": chunk.usage.completion_tokens,
                "total_tokens": chunk.usage.total_tokens,
            })

    # 3) 끝을 알리는 줄(OpenAI, vLLM 과 같은 약속)
    yield "data: [DONE]\n\n"
```

```python
# routers/labor_router.py
@labor_router.post("/chat/stream")
async def labor_chat_stream(request: ChatRequest):
    """노무 질문의 답을 조각이 만들어지는 대로 SSE(text/event-stream)로 보낸다."""
    # 스트림을 응답보다 먼저 연다. 첫 조각을 보낸 뒤에는 상태 코드를 바꿀 수 없으므로, vLLM 에 닿지 않으면 응답을 시작하기 전에 여기서 오류를 낸다
    stream = await open_vllm_stream(LABOR_SYSTEM_PROMPT, request.question, request.think, max_tokens_think=settings.max_tokens_think_labor)
    # relay_stream 이 yield 한 줄을 바로 보내고, media_type 으로 SSE 응답임을 알린다
    return StreamingResponse(relay_stream(stream), media_type="text/event-stream")
```

### 6-2. SSE 줄 그대로 보기

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">httpx.stream</code>은 응답 본문을 한꺼번에 받지 않고 도착하는 대로 읽게 하고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">iter_lines()</code>는 받은 본문을 한 줄씩 꺼냅니다. 먼저 서빙 API가 보낸 줄을 바꾸지 않고 모아 봅니다.</p>
</div>

In [ ]:
# 스트리밍 주소에 노무 질문을 보내고, 받은 줄을 바꾸지 않고 모읍니다
raw_lines = []
with httpx.stream("POST", f"{SERVING_URL}/v1/labor/chat/stream", json={"question": user_question}, timeout=60) as response:
    for line in response.iter_lines():
        raw_lines.append(line)

# 앞 5줄과 끝 5줄을 봅니다(가운데에는 답 조각 줄이 이어집니다)
raw_view = {"전체 줄 수": len(raw_lines), "앞 5줄": raw_lines[:5], "끝 5줄": raw_lines[-5:]}
display(raw_view)

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">data:</code> 줄과 빈 줄(<code style="color:inherit;background:#f3f6fa;font-size:0.95em;">''</code>):</strong> 이벤트마다 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">data: {...}</code> 한 줄과 빈 줄 하나가 번갈아 옵니다. 생각을 껐으므로 앞부분의 이벤트는 모두 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;type&quot;: &quot;answer&quot;</code>입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">끝 5줄:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;type&quot;: &quot;usage&quot;</code> 줄에 토큰 사용량이 있고, 그 뒤의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">data: [DONE]</code>이 끝 표시입니다.</li>
</ul>
</div>

### 6-3. 조각 도착 시각과 TTFT

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">아래 두 함수는 함께 따라하기에서 생각을 켠 요청에도 쓰므로 함수로 둡니다.</p>
</div>

In [ ]:
def read_stream(path, question, think):
    """서빙 API의 스트리밍 주소에 질문을 보내고, 받은 이벤트마다 도착 시각(ms)을 붙여 리스트로 돌려줍니다."""
    events = []
    shown_type = None
    reasoning_chars = 0
    sent_at = time.perf_counter()
    with httpx.stream("POST", f"{SERVING_URL}{path}", json={"question": question, "think": think}, timeout=60) as response:
        # 200이 아니면 조각을 읽기 전에 오류로 멈춥니다
        response.raise_for_status()
        for line in response.iter_lines():
            # 이벤트는 "data: "로 시작하는 줄에 하나씩 오고, 이벤트 사이의 빈 줄은 건너뜁니다
            if not line.startswith("data: "):
                continue
            payload = line.removeprefix("data: ")

            # 끝을 알리는 줄이면 읽기를 마칩니다
            if payload == "[DONE]":
                break

            # 이벤트 딕셔너리에 보낸 때부터의 도착 시각을 붙여 모읍니다
            event = json.loads(payload)
            event["arrived_ms"] = round((time.perf_counter() - sent_at) * 1000, 1)
            events.append(event)

            # 생각 조각과 답 조각은 도착하는 대로 화면에 씁니다. 종류가 바뀌는 자리에 [reasoning], [answer] 표시를 붙입니다
            if event["type"] != "usage":
                if event["type"] != shown_type:
                    print(f"\n[{event['type']}]", end=" ")
                    shown_type = event["type"]

                # 생각은 1만 자를 넘을 수 있어 앞 500자 안쪽까지만 씁니다(전체 글자 수는 summarize_events 로 봅니다)
                if event["type"] == "reasoning":
                    reasoning_chars += len(event["text"])
                    if reasoning_chars > 500:
                        continue
                print(event["text"], end="", flush=True)
    return events


def summarize_events(events):
    """이벤트 리스트에서 조각 수, 생각 글자 수, 첫 조각과 답 조각의 도착 시각(ms), 출력 토큰 수를 딕셔너리로 모읍니다."""
    reasoning_count = 0
    reasoning_chars = 0
    answer_times = []
    completion_tokens = None
    for event in events:
        if event["type"] == "reasoning":
            reasoning_count += 1
            reasoning_chars += len(event["text"])
        elif event["type"] == "answer":
            answer_times.append(event["arrived_ms"])
        elif event["type"] == "usage":
            completion_tokens = event["completion_tokens"]

    # 생각이 생성 상한을 다 쓰면 답 조각이 없으므로, 그때는 답 조각 시각을 None으로 둡니다
    first_answer_ms = None
    last_answer_ms = None
    if answer_times:
        first_answer_ms = answer_times[0]
        last_answer_ms = answer_times[-1]

    return {
        "생각 조각 수": reasoning_count,
        "생각 글자 수": reasoning_chars,
        "답 조각 수": len(answer_times),
        "첫 조각(ms, TTFT)": events[0]["arrived_ms"],
        "첫 답 조각(ms)": first_answer_ms,
        "마지막 답 조각(ms)": last_answer_ms,
        "출력 토큰 수": completion_tokens,
    }

In [ ]:
# 생각을 끈 스트리밍 요청: 답 조각이 도착하는 대로 화면에 찍힙니다
labor_events = read_stream("/v1/labor/chat/stream", user_question, False)

In [ ]:
# 조각 수와 첫 조각, 마지막 답 조각의 도착 시각
display(summarize_events(labor_events))

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">첫 조각(ms, TTFT)</code>:</strong> 노트북이 요청을 보낸 때부터 첫 조각이 도착할 때까지입니다. 서빙 벤치마킹 단원의 TTFT와 같은 정의이고, FastAPI 서버를 거친 시간이 함께 들어 있습니다. 생각을 껐으므로 첫 조각이 곧 첫 답 조각입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">마지막 답 조각(ms)</code>:</strong> 답 전체를 받은 시각입니다. 4-3의 일반 응답은 이 시각까지 아무것도 받지 못하고, 스트리밍은 첫 조각부터 답을 화면에 보입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">생각 조각 수</code>, <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">생각 글자 수</code>:</strong> 생각을 껐으므로 0입니다. 생각을 켜면 생각 조각이 먼저 오는 것은 7-3에서 전월세 창구로 확인합니다.</li>
</ul>
</div>

## 7. 🖐️ 함께 따라하기: 전월세 상담 창구 열기

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">본실습의 노무 창구와 같은 모양으로 전월세 창구(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1/lease</code>)의 두 엔드포인트 몸통을 직접 작성하고, 노트북에서 일반 응답과 생각을 켠 스트리밍 응답을 확인합니다. 전월세 창구는 시스템 메시지가 약 1,800토큰으로 짧아 생각 상한 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">max_tokens_think</code>(4400)를 그대로 씁니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">본실습과 다른 점</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">서버 파일:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving_app/routers/lease_router.py</code>의 두 엔드포인트는 함수 원형(데코레이터, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">async def</code> 줄, 설명 문자열)만 있고 몸통이 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">...</code>입니다. 묶음(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">lease_router</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">prefix=&quot;/v1/lease&quot;</code>)과 시스템 메시지(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">LEASE_SYSTEM_PROMPT</code>, 주택임대차보호법 참고 조문)는 완성되어 있습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">질문:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">data/따라하기/lease_questions.jsonl</code>(질문 73개)</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">주소:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">POST /v1/lease/chat</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">POST /v1/lease/chat/stream</code></li>
</ul>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">실습 진행 방법:</strong> .py 파일과 코드 셀의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">...</code> 줄을 바로 위 주석이 말하는 코드로 바꿉니다. 인자 몇 개만 채우는 것이 아니라, 부를 함수와 넘길 값, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">await</code>, 결과를 담을 변수나 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">return</code>까지 한 줄을 통째로 씁니다. 생각을 켠 요청은 답까지 1분 넘게 걸릴 수 있습니다. 본실습의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">load_jsonl</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">read_stream</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">summarize_events</code>를 그대로 쓰므로 본실습을 먼저 실행합니다. 완성 코드는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">정답/serving_app/routers/lease_router.py</code>와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">정답/교안_01_정답.ipynb</code>의 같은 절에 있습니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">설정</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">vLLM 서버(터미널 1)와 FastAPI 서버(터미널 2)는 본실습에서 띄운 것을 그대로 씁니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--reload</code>로 띄웠으므로 파일을 저장하면 FastAPI 서버가 다시 시작합니다.</li>
</ul>
</div>

### 7-1. routers/lease_router.py 몸통 작성

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">엔드포인트 몸통은 두 단계입니다. 요청 양식(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">request</code>)의 값을 서비스 계층(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">services/</code>)에 넘겨 vLLM의 결과를 받고, 그 결과를 응답으로 바꿔 돌려줍니다. 일반 응답은 응답 양식(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ChatResponse</code>)으로, 스트리밍 응답은 SSE 스트림으로 돌려줍니다. 쓸 함수는 파일 위쪽 import 줄에 모두 들어 있습니다.</p>
<p style="margin:6px 0;">VS Code 편집기에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving_app/routers/lease_router.py</code>를 열어 주석 (1)~(4) 아래의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">...</code>를 코드로 바꾸고 저장합니다. 작성할 부분은 아래와 같습니다.</p>
</div>

```python
# routers/lease_router.py
@lease_router.post("/chat", response_model=ChatResponse)
async def lease_chat(request: ChatRequest):
    """전월세 질문 하나에 대한 답 전체를 한 번에 돌려준다."""
    # 🖐️ 함께 따라하기 (1): 전월세 시스템 메시지, 요청의 질문, 요청의 생각 켜기 여부로 vLLM 에 묻고, 답이 다 만들어질 때까지 기다린다
    ...

    # 🖐️ 함께 따라하기 (2): vLLM 의 응답(ChatCompletion)을 서빙 API 의 응답 양식(ChatResponse)으로 바꿔 돌려준다
    ...


@lease_router.post("/chat/stream")
async def lease_chat_stream(request: ChatRequest):
    """전월세 질문의 답을 조각이 만들어지는 대로 SSE(text/event-stream)로 보낸다."""
    # 🖐️ 함께 따라하기 (3): 일반 응답과 같은 세 값으로 vLLM 스트림을 연다. 응답을 시작하기 전에 열어야 vLLM 에 닿지 않을 때 바로 오류로 알린다
    ...

    # 🖐️ 함께 따라하기 (4): 스트림의 조각을 SSE 이벤트로 바꿔 보내는 응답(text/event-stream)을 돌려준다
    ...
```

<details style="margin:8px 0;"><summary style="font-size:17px;color:#1f3a5f;font-weight:600;cursor:pointer;margin:6px 0;">힌트</summary>

```text
접근방법: 3-1, 4-2, 6-1의 routers/labor_router.py에서 labor_chat, labor_chat_stream의 몸통을 한 줄씩 비교합니다. 노무 창구와 다른 점은 시스템 메시지가 전월세 시스템 메시지이고, 생각 상한 인자(max_tokens_think)를 넘기지 않는다는 것입니다.
(1) ask_vllm에 시스템 메시지, 질문, 생각 켜기 여부를 이 순서로 넘깁니다. 질문과 생각 켜기 여부는 요청 양식(request)의 두 항목입니다. vLLM의 답을 기다려야 하므로 await로 받아 변수에 담습니다.
(2) (1)의 변수를 to_chat_response로 바꾼 값을 return합니다.
(3) open_vllm_stream도 (1)과 같은 세 값을 받고, await로 받은 스트림을 변수에 담습니다.
(4) StreamingResponse에 relay_stream으로 감싼 스트림과, 6-1에서 본 SSE의 media_type 문자열을 넘겨 return합니다.
```

</details>

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">확인 기준:</strong> 파일을 저장하면 터미널 2에 FastAPI 서버가 다시 시작하는 줄과 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Application startup complete</code>가 다시 나오고, 오류 줄(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Traceback</code>)이 없습니다. 몸통이 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">...</code>인 채로 저장해도 서버는 뜨므로, 작성 여부는 7-2와 7-3의 응답으로 확인합니다.</p>
</div>

### 7-2. 전월세 질문 보내기

In [ ]:
# 전월세 상담 질문(id, 질문, 출처 주제)을 읽어 앞 3개를 봅니다
lease_questions = load_jsonl(data_dir / "따라하기" / "lease_questions.jsonl")
display(pd.DataFrame(lease_questions).head(3))

<details style="margin:8px 0;"><summary style="font-size:17px;color:#1f3a5f;font-weight:600;cursor:pointer;margin:6px 0;">힌트</summary>

```text
접근방법: 4-3을 참고합니다.
httpx.post로 보냅니다. 주소는 SERVING_URL 뒤에 lease_router의 prefix와 일반 응답 엔드포인트의 주소를 이은 f-문자열입니다.
요청 본문(json)은 question 항목에 바로 위 줄의 lease_question을 담은 딕셔너리이고, timeout은 60초로 둡니다.
```

</details>

In [ ]:
# 첫 번째 전월세 질문
lease_question = lease_questions[0]["question"]

# 🖐️ 함께 따라하기: 전월세 창구의 일반 응답 주소로 질문을 담은 POST 요청을 보내고, 응답을 response 에 담습니다
response = ...

# 200이 아니면 여기서 오류로 멈춥니다(몸통을 채우지 않은 서버 파일은 500을 돌려줍니다)
response.raise_for_status()
lease_reply = response.json()
display(lease_reply)

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">확인 기준:</strong> 오류 없이 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">lease_reply</code>가 출력되고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">reasoning</code>이 None이며 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">answer</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">finish_reason</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">model</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">usage</code>가 채워져 있습니다.</p>
</div>

### 7-3. 생각을 켠 스트리밍 응답

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">같은 질문을 스트리밍 주소에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">think=True</code>로 보냅니다. 서버는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">enable_thinking</code>을 켜고 생성 상한을 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">max_tokens_think</code>로 올립니다. 생각은 영어로 나올 수 있고 답은 한국어이며, 모델 카드 샘플링(4-2)을 쓰므로 생각의 길이와 답이 실행마다 다릅니다. 화면에는 생각의 앞 500자만 찍으므로, 그 뒤 답 조각이 올 때까지 1분 가까이 아무것도 찍히지 않을 수 있습니다. 셀 왼쪽의 실행 표시가 돌고 있으면 멈춘 것이 아니니 기다립니다.</p>
</div>

<details style="margin:8px 0;"><summary style="font-size:17px;color:#1f3a5f;font-weight:600;cursor:pointer;margin:6px 0;">힌트</summary>

```text
접근방법: 6-3을 참고합니다.
본실습의 read_stream에 스트리밍 경로, 질문, 생각 켜기 여부를 이 순서로 넘기고, 돌려받은 이벤트 리스트를 lease_events에 담습니다.
read_stream은 SERVING_URL 뒤의 경로만 받습니다(6-3의 /v1/labor/chat/stream처럼). 전월세 창구의 경로는 prefix /v1/lease 뒤에 스트리밍 엔드포인트의 주소를 이은 것이고, 질문은 7-2의 lease_question입니다.
```

</details>

In [ ]:
# 🖐️ 함께 따라하기: 전월세 창구의 스트리밍 주소로 7-2의 질문을 생각을 켜서 보내고, 받은 이벤트 리스트를 lease_events 에 담습니다
lease_events = ...

In [ ]:
# 전월세 스트리밍의 조각 수, 생각 글자 수, 도착 시각
display(summarize_events(lease_events))

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">첫 조각(ms, TTFT)</code>과 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">첫 답 조각(ms)</code>:</strong> 생각을 켜면 첫 조각은 생각 조각이고, 첫 답 조각은 생각 조각이 모두 끝난 뒤에 옵니다. TTFT가 6-3의 생각을 끈 요청과 비슷해도, 사용자가 답을 읽기 시작하는 시각은 첫 답 조각입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">생각 글자 수</code>:</strong> 화면에는 앞 500자 안쪽만 찍혔고, 이 값이 생각 전체의 글자 수입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">출력 토큰 수</code>:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">completion_tokens</code>에는 생각 토큰도 들어 있어, 생각을 켜면 답만 만들 때보다 훨씬 큽니다. 응답의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">reasoning</code>과 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">answer</code>를 나눠 받아도 사용량은 둘을 합해 셉니다.</li>
</ul>
<p style="margin:6px 0;">생각을 켠 창구는 첫 답 조각까지 오래 걸립니다. 그래서 생각 조각을 화면에 따로 보여 주어 기다림을 덜고, 빠른 답이 필요한 창구는 생각을 끈 채 서비스합니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">확인 기준:</strong> 화면에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">[reasoning]</code> 조각 다음에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">[answer]</code> 조각이 이어 찍히고, 요약의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">생각 조각 수</code>와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">답 조각 수</code>가 모두 0보다 큽니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">출력 토큰 수</code>가 4400이면 생각과 답이 생성 상한에 닿은 것입니다. 이때 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">답 조각 수</code>가 0이면 답을 시작하지 못한 것이고, 0보다 크면 답이 중간에서 잘린 것입니다(스트리밍 이벤트에는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">finish_reason</code>이 없어 출력 토큰 수로 판단합니다).</p>
<p style="margin:6px 0;">요약 셀에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">IndexError</code>가 나면 받은 이벤트가 하나도 없는 것입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">routers/lease_router.py</code>의 (3), (4)를 채우지 않으면 스트리밍 주소가 이벤트 대신 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">null</code>을 돌려줍니다.</p>
</div>

## 8. 핵심 정리

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">실행 위치:</strong> 같은 RunPod 파드에서 터미널 1은 vLLM(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">127.0.0.1:8000</code>), 터미널 2는 FastAPI(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">127.0.0.1:8080</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">uv run uvicorn serving_app.main:app --host 0.0.0.0 --port 8080 --reload</code>)를 띄우고, 원격 노트북은 FastAPI에만 요청합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">계층 구조와 라우팅:</strong> 서버 코드를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving_app/</code> 패키지의 계층 폴더(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">routers/</code> 주소, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">services/</code> vLLM 호출, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">schemas/</code> 양식, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">core/</code> 설정)로 나누고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">APIRouter(prefix=..., tags=[...])</code>로 창구별 엔드포인트를 묶어 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">app.include_router</code>로 앱에 붙입니다. 실제 주소는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">prefix</code> 뒤에 엔드포인트 주소를 이은 것입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">vLLM 백엔드 연동:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ChatRequest</code>는 질문과 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">think</code>만 받고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ChatResponse</code>는 답과 생각을 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">answer</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">reasoning</code>에 나눠 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">finish_reason</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">model</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">usage</code>와 함께 돌려줍니다. 서버가 뜰 때 만든 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">AsyncOpenAI</code> 하나로 vLLM에 요청하고, 시스템 메시지와 생성 상한은 서버가 붙입니다. 생각 상한은 입력 토큰 수와 합쳐 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--max-model-len</code>(8192)을 넘지 않게 창구마다 정합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">동시 요청:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">async def</code> 엔드포인트가 vLLM의 답을 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">await</code>로 기다리는 동안 다른 요청을 받고, vLLM은 함께 들어온 요청을 묶어 계산합니다. 노트북은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">asyncio.Semaphore</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">run_limited</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">asyncio.gather</code>로 요청을 동시에 보냅니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">스트리밍 중계:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">relay_stream</code>이 vLLM의 OpenAI 방식 조각을 서빙 API의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">type</code>(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">reasoning</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">answer</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">usage</code>) 이벤트를 담은 SSE <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">data:</code> 줄로 바꾸고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">StreamingResponse</code>가 그 줄을 받는 대로 보냅니다. 노트북은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">httpx.stream</code>과 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">iter_lines()</code>로 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">data:</code> 줄을 읽고, 첫 조각의 도착 시각으로 FastAPI를 거친 TTFT를 계산합니다.</li>
</ul>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">서버는 켜 둡니다:</strong> 터미널 1의 vLLM 서버는 끄지 않고 오늘 끝까지 씁니다. 터미널 2의 FastAPI 서버는 다음 노트북(과제 01, 교안 02)의 준비 블록에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code>로 끄고, 그 폴더의 앱(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop_api.main</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving_app.secure_main</code>)으로 바꿔 띄웁니다. 오늘 실습을 여기서 마치면 아래 접힌 블록을 따라 두 서버를 끕니다.</p>
</div>

<details style="margin:8px 0;"><summary style="font-size:17px;color:#1f3a5f;font-weight:600;cursor:pointer;margin:6px 0;">오늘 실습을 여기서 마칠 때</summary>

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">RunPod 터미널 2</strong>(FastAPI 서버)와 <strong style="font-size:1.13em;color:#1f3a5f;">RunPod 터미널 1</strong>(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">qwen3.5-4b</code> vLLM 서버)에서 차례로 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code> 키</strong>를 누릅니다. 명령을 입력할 수 있게 되면 터미널 1에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">nvidia-smi</code>를 실행합니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">GPU 메모리 사용량(<code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Memory-Usage</code>)이 서버를 띄우기 전 수준으로 줄었으면</strong> vLLM 서버가 꺼진 것입니다.</p>
<p style="margin:6px 0;">오늘 실습을 마치면 RunPod 콘솔에서 파드를 종료(Terminate)합니다. 정지(Stop)해도 다음에 같은 GPU로 다시 켜지지 않는 경우가 많습니다. 남길 파일(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output</code> 등)이 있으면 종료 전에 내 PC로 받아 둡니다.</p>
</div>

</details>